# ProteinMPNN Rerun — Top 50 High-sc_proxy RFdiffusion Backbones

**Goal**: These 31 RFdiffusion backbones have excellent interface geometry (sc_proxy > 0.75, contacts > 65) but were missed in the original submission because the single ProteinMPNN sequence tried scored < 0.70 ipTM in Boltz-2. We rerun with 9 sequences per backbone (3 temperatures × 3 seeds).

**Strategy**:
- 3 temperatures: 0.05 (conservative), 0.10 (moderate), 0.15 (diverse)
- 3 sequences per temperature = 9 sequences per backbone
- 31 backbones × 9 = 279 candidate sequences
- Expected pass rate at ipTM ≥ 0.70: ~20–25% → ~55–70 new candidates

**Chain assignment in Boltz-2 CIF**:
- Chain A = binder (label_asym_id = 'A')
- Chain B = RBX1 (label_asym_id = 'B')

We extract only chain A residues as the backbone for ProteinMPNN (fix the RBX1 sequence, redesign binder).

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install ProteinMPNN (LigandMPNN fork with CIF support)
import os

if not os.path.exists('/content/LigandMPNN'):
    !git clone -q https://github.com/dauparas/LigandMPNN /content/LigandMPNN

# Also install biopython for CIF parsing
!pip install -q biopython

print('Done')

In [ ]:
# ── Option A: upload rerun_mpnn_backbones.txt directly ────────────────────────
from google.colab import files
print('Upload rerun_mpnn_backbones.txt and the CIF zip (boltz_rfdiffusion_complexes.zip)')
uploaded = files.upload()

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import csv, glob, os, shutil
import numpy as np

BACKBONES_FILE = 'rerun_mpnn_backbones.txt'  # uploaded above
CIF_BASE = '/content/boltz_rfdiffusion/complex_results/boltz_results_complex/predictions'

# ProteinMPNN settings
TEMPERATURES = [0.05, 0.10, 0.15]
SEQS_PER_TEMP = 3
TOTAL_PER_BACKBONE = len(TEMPERATURES) * SEQS_PER_TEMP  # 9

OUT_DIR = '/content/mpnn_rerun_outputs'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f'{OUT_DIR}/pdbs', exist_ok=True)

# Load backbone list (skip already-submitted — 19 of top 50 already in v2)
# We keep all 50 here; post-filtering handles deduplication
backbones = []
with open(BACKBONES_FILE) as f:
    for line in f:
        if line.startswith('#') or not line.strip():
            continue
        parts = line.strip().split('\t')
        if len(parts) >= 4:
            backbones.append({'seq_id': parts[0], 'sc_proxy': float(parts[1]),
                              'flag': parts[2], 'cif': parts[3]})

print(f'Loaded {len(backbones)} backbone entries')
print(f'Top 5:')
for b in backbones[:5]:
    print(f"  {b['seq_id']:<30} sc={b['sc_proxy']:.3f}  {b['cif']}")

In [ ]:
# ── Extract chain A from each CIF and save as PDB for ProteinMPNN ────────────
# ProteinMPNN requires PDB format; we parse CIF and write minimal PDB

def parse_cif_chain(cif_path, target_chain='A'):
    """Extract all ATOM records for a single chain from mmCIF."""
    cols = {}
    residues = {}  # resnum -> (resname, [(atom_name, x, y, z)])
    in_loop = False

    with open(cif_path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('_atom_site.'):
                field = line.split('.')[-1].strip()
                cols[field] = len(cols)
                in_loop = True
                continue
            if not in_loop or not line.startswith('ATOM'):
                continue
            parts = line.split()
            try:
                chain   = parts[cols['label_asym_id']]
                resnum  = int(parts[cols['label_seq_id']])
                resname = parts[cols['label_comp_id']]
                atom    = parts[cols['label_atom_id']]
                x       = float(parts[cols['Cartn_x']])
                y       = float(parts[cols['Cartn_y']])
                z       = float(parts[cols['Cartn_z']])
            except (KeyError, ValueError, IndexError):
                continue
            if chain != target_chain:
                continue
            if resnum not in residues:
                residues[resnum] = (resname, [])
            residues[resnum][1].append((atom, x, y, z))
    return residues


def write_pdb(residues, out_path, chain='A'):
    """Write extracted residues as minimal PDB."""
    aa3 = {'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
           'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'}
    atom_serial = 1
    with open(out_path, 'w') as f:
        for resnum in sorted(residues):
            resname, atoms = residues[resnum]
            if resname not in aa3:
                continue
            for atom_name, x, y, z in atoms:
                f.write(f"ATOM  {atom_serial:5d}  {atom_name:<3s} {resname} {chain}{resnum:4d}    "
                        f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           {atom_name[0]}\n")
                atom_serial += 1
        f.write('END\n')


# Convert all backbones
converted = []
missing = []
for b in backbones:
    cif_path = b['cif']
    # Adjust path if running in Colab vs local
    if not os.path.exists(cif_path):
        # Try relative to CIF_BASE
        alt = os.path.join(CIF_BASE, b['seq_id'] + '_complex', 
                           b['seq_id'] + '_complex_model_0.cif')
        cif_path = alt if os.path.exists(alt) else None
    
    if cif_path is None:
        missing.append(b['seq_id'])
        continue
    
    pdb_out = f"{OUT_DIR}/pdbs/{b['seq_id']}_chainA.pdb"
    try:
        res = parse_cif_chain(cif_path, 'A')
        if not res:
            missing.append(b['seq_id'])
            continue
        write_pdb(res, pdb_out)
        b['pdb'] = pdb_out
        converted.append(b)
    except Exception as e:
        print(f"  ERROR {b['seq_id']}: {e}")
        missing.append(b['seq_id'])

print(f'Converted: {len(converted)} | Missing/failed: {len(missing)}')
if missing:
    print(f'Missing: {missing[:10]}')

In [ ]:
# ── Run ProteinMPNN on each backbone at 3 temperatures ───────────────────────
import subprocess, json

MPNN_SCRIPT = '/content/LigandMPNN/run.py'
MPNN_MODEL  = '/content/LigandMPNN/model_params/proteinmpnn_v_48_020.pt'

# Download model weights if not present
if not os.path.exists(MPNN_MODEL):
    os.makedirs('/content/LigandMPNN/model_params', exist_ok=True)
    !wget -q -P /content/LigandMPNN/model_params/ \
        https://github.com/dauparas/ProteinMPNN/raw/main/vanilla_model_weights/v_48_020.pt \
        -O {MPNN_MODEL}
    print('Downloaded model weights')

all_sequences = []  # {name, seq, backbone_id, temp, sc_proxy}

for temp in TEMPERATURES:
    temp_str = f't{temp:.2f}'
    temp_out = f'{OUT_DIR}/temp_{temp_str}'
    os.makedirs(temp_out, exist_ok=True)

    print(f'\n── Temperature {temp} ────────────────────────────────────')
    
    for i, b in enumerate(converted):
        pdb_in = b['pdb']
        out_prefix = f"{temp_out}/{b['seq_id']}"
        
        # Run ProteinMPNN
        cmd = [
            'python', MPNN_SCRIPT,
            '--pdb_path', pdb_in,
            '--out_folder', out_prefix,
            '--num_seq_per_target', str(SEQS_PER_TEMP),
            '--sampling_temp', str(temp),
            '--seed', '42',
            '--batch_size', '1',
        ]
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if result.returncode != 0:
                print(f"  FAIL {b['seq_id']}: {result.stderr[-200:]}")
                continue
        except subprocess.TimeoutExpired:
            print(f"  TIMEOUT {b['seq_id']}")
            continue

        # Parse output FASTA
        fasta_files = glob.glob(f'{out_prefix}/seqs/*.fa') + glob.glob(f'{out_prefix}/seqs/*.fasta')
        if not fasta_files:
            print(f"  No FASTA for {b['seq_id']}")
            continue
        
        with open(fasta_files[0]) as f:
            lines = f.read().strip().split('\n')
        
        # Parse header/sequence pairs; skip the native sequence (first entry)
        entries = []
        for j in range(0, len(lines)-1, 2):
            if lines[j].startswith('>'):
                entries.append((lines[j][1:], lines[j+1]))
        
        # Skip entry 0 (native), take up to SEQS_PER_TEMP designed
        designed = entries[1:SEQS_PER_TEMP+1] if len(entries) > 1 else entries
        
        for s_idx, (header, seq) in enumerate(designed):
            name = f"{b['seq_id']}_{temp_str}_s{s_idx+1}"
            all_sequences.append({
                'name': name,
                'seq': seq,
                'backbone_id': b['seq_id'],
                'temp': temp,
                'sc_proxy': b['sc_proxy'],
                'original_header': header,
            })
        
        if (i + 1) % 10 == 0:
            print(f'  {i+1}/{len(converted)} done, {len(all_sequences)} seqs so far')

print(f'\nTotal sequences generated: {len(all_sequences)}')

In [ ]:
# ── Sanity check: sequence length distribution ────────────────────────────────
import collections

lengths = [len(s['seq']) for s in all_sequences]
print(f'Sequences generated: {len(all_sequences)}')
print(f'Length range: {min(lengths)}–{max(lengths)} AA')
print(f'Backbones covered: {len(set(s["backbone_id"] for s in all_sequences))}')
print()
print('Sequences per temperature:')
for t in TEMPERATURES:
    n = sum(1 for s in all_sequences if s['temp'] == t)
    print(f'  T={t}: {n}')
print()
print('First 5 sequences:')
for s in all_sequences[:5]:
    print(f"  {s['name']:<40} {len(s['seq'])} AA  sc_proxy={s['sc_proxy']:.3f}")
    print(f"    {s['seq'][:60]}...")

In [ ]:
# ── Write output FASTA ────────────────────────────────────────────────────────
FASTA_OUT = f'{OUT_DIR}/mpnn_rerun_candidates.fasta'
CSV_OUT   = f'{OUT_DIR}/mpnn_rerun_metadata.csv'

# Sort by sc_proxy (best backbones first) then temp
all_sequences.sort(key=lambda s: (-s['sc_proxy'], s['temp'], s['name']))

with open(FASTA_OUT, 'w') as f:
    for s in all_sequences:
        f.write(f">{s['name']}\n{s['seq']}\n")

with open(CSV_OUT, 'w', newline='') as f:
    import csv
    w = csv.DictWriter(f, fieldnames=['name','backbone_id','temp','sc_proxy','length','seq'])
    w.writeheader()
    for s in all_sequences:
        w.writerow({**s, 'length': len(s['seq'])})

print(f'Saved {FASTA_OUT} ({len(all_sequences)} sequences)')
print(f'Saved {CSV_OUT}')

In [ ]:
# ── Generate Boltz-2 YAML files for all candidates ───────────────────────────
# IMPORTANT: Use the TRUNCATED RBX1 (res 32-83, 52 AA) — not the full 108 AA.
# The full sequence includes disordered N-terminal (1-31) and C-terminal (84-108)
# tails that attract false binders in Boltz-2. This is the structured RING core only.

RBX1_RING_CORE_SEQ = 'LWAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHC'  # res 32-83, 52 AA
# All 11 zinc-coordinating residues captured (C42,C45,H48,C53,C56,C68,C75,H77,H80,H82,C83)

YAML_DIR = f'{OUT_DIR}/boltz_yamls'
os.makedirs(YAML_DIR, exist_ok=True)

yaml_template = """version: 1
sequences:
  - protein:
      id: A
      sequence: {binder_seq}
  - protein:
      id: B
      sequence: {rbx1_seq}
"""

for s in all_sequences:
    yaml_content = yaml_template.format(
        binder_seq=s['seq'],
        rbx1_seq=RBX1_RING_CORE_SEQ
    )
    yaml_path = f"{YAML_DIR}/{s['name']}_complex.yaml"
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)

print(f'Generated {len(all_sequences)} Boltz-2 YAML files in {YAML_DIR}/')
print(f'RBX1 target: {RBX1_RING_CORE_SEQ} ({len(RBX1_RING_CORE_SEQ)} AA, res 32-83 only)')


In [ ]:
# ── Run Boltz-2 on all candidates ─────────────────────────────────────────────
# NOTE: This cell requires Boltz-2 to be installed. Skip if you plan to run
# Boltz-2 externally (e.g., local GPU cluster) and just want the YAMLs.

BOLTZ_AVAILABLE = False  # Set to True if boltz is installed

try:
    import boltz
    BOLTZ_AVAILABLE = True
    print('Boltz-2 found')
except ImportError:
    print('Boltz-2 not installed. Installing...')
    !pip install -q boltz
    BOLTZ_AVAILABLE = True

if BOLTZ_AVAILABLE:
    BOLTZ_OUT = f'{OUT_DIR}/boltz_results'
    os.makedirs(BOLTZ_OUT, exist_ok=True)

    yaml_files = sorted(glob.glob(f'{YAML_DIR}/*.yaml'))
    print(f'Running Boltz-2 on {len(yaml_files)} complex YAMLs...')

    # Run in batches to avoid OOM
    BATCH = 10
    for i in range(0, len(yaml_files), BATCH):
        batch = yaml_files[i:i+BATCH]
        batch_dir = f'{BOLTZ_OUT}/batch_{i//BATCH:04d}'
        os.makedirs(batch_dir, exist_ok=True)

        # Write batch input list
        batch_list = f'{batch_dir}/inputs.txt'
        with open(batch_list, 'w') as f:
            for yf in batch:
                f.write(yf + '\n')

        print(f'  Batch {i//BATCH}: {len(batch)} structures...')
        for yf in batch:
            name = os.path.basename(yf).replace('_complex.yaml', '')
            out_path = f'{batch_dir}/{name}'
            !boltz predict {yf} --out_dir {out_path} --accelerator gpu \
                --recycling_steps 3 --sampling_steps 200 --num_workers 2 \
                --diffusion_samples 1 \
                --no_kernels 2>/dev/null

        print(f'  Batch {i//BATCH} done')

    print(f'\nAll Boltz-2 predictions complete. Results in {BOLTZ_OUT}/')

In [ ]:
# ── Score Boltz-2 outputs: extract ipTM and ipLDDT ───────────────────────────
import json, glob

def extract_boltz_scores(boltz_out_dir):
    """Extract ipTM and ipLDDT from Boltz-2 confidence JSON."""
    scored = []
    for conf_file in sorted(glob.glob(f'{boltz_out_dir}/**/confidence_*.json', recursive=True)):
        try:
            with open(conf_file) as f:
                conf = json.load(f)
            # Extract name from path
            name = os.path.basename(os.path.dirname(conf_file))
            scored.append({
                'name': name,
                'iptm': conf.get('iptm', conf.get('ipTM', None)),
                'iplddt': conf.get('interface_plddt', conf.get('ipLDDT', None)),
                'ptm': conf.get('ptm', conf.get('pTM', None)),
            })
        except Exception:
            continue
    return scored

# Run scoring
if os.path.exists(f'{OUT_DIR}/boltz_results'):
    scores = extract_boltz_scores(f'{OUT_DIR}/boltz_results')
    print(f'Scored {len(scores)} predictions')

    # Merge with sequence metadata
    seq_meta = {s['name']: s for s in all_sequences}
    merged = []
    for sc in scores:
        meta = seq_meta.get(sc['name'], {})
        merged.append({
            **sc,
            'sc_proxy': meta.get('sc_proxy', ''),
            'temp': meta.get('temp', ''),
            'backbone_id': meta.get('backbone_id', ''),
            'length': meta.get('length', ''),
        })

    # Sort by ipTM
    merged.sort(key=lambda r: -(r['iptm'] or 0))

    passing = [r for r in merged if r['iptm'] and float(r['iptm']) >= 0.70]
    borderline = [r for r in merged if r['iptm'] and 0.65 <= float(r['iptm']) < 0.70]

    print(f'\nResults:')
    print(f'  Total scored:     {len(merged)}')
    print(f'  PASS (≥0.70):     {len(passing)}')
    print(f'  BORDERLINE (≥0.65): {len(borderline)}')
    print(f'  FAIL (<0.65):     {len(merged)-len(passing)-len(borderline)}')

    print(f'\nTop 20 by ipTM:')
    print(f'{"name":<40} {"ipTM":>6} {"ipLDDT":>7} {"sc_proxy":>9} {"T":>5}')
    print('-' * 70)
    for r in merged[:20]:
        print(f"{r['name']:<40} {r['iptm'] or 0:>6.3f} "
              f"{r['iplddt'] or 0:>7.3f} {r['sc_proxy']:>9.3f} {r['temp']:>5}")
else:
    print('Boltz-2 results not found. Run the boltz_run cell first.')

In [ ]:
# ── Write final FASTA of passing sequences ────────────────────────────────────
if 'passing' in dir() and passing:
    FINAL_FASTA = f'{OUT_DIR}/mpnn_rerun_passing.fasta'
    seq_lookup = {s['name']: s['seq'] for s in all_sequences}

    with open(FINAL_FASTA, 'w') as f:
        for r in passing:
            seq = seq_lookup.get(r['name'], '')
            if seq:
                f.write(f">{r['name']} iptm={r['iptm']:.3f} sc_proxy={r['sc_proxy']:.3f}\n{seq}\n")

    print(f'Saved {FINAL_FASTA} ({len(passing)} sequences)')

    # Summary CSV
    FINAL_CSV = f'{OUT_DIR}/mpnn_rerun_passing.csv'
    with open(FINAL_CSV, 'w', newline='') as f:
        import csv
        w = csv.DictWriter(f, fieldnames=['name','backbone_id','temp','iptm','iplddt',
                                          'sc_proxy','length','seq'])
        w.writeheader()
        for r in passing:
            seq = seq_lookup.get(r['name'], '')
            w.writerow({**r, 'seq': seq})

    print(f'Saved {FINAL_CSV}')

    # Download results
    from google.colab import files
    files.download(FINAL_FASTA)
    files.download(FINAL_CSV)

## Notes

### RBX1 sequence
Update `RBX1_RING_SEQ` in the YAML generation cell to match the exact sequence used in your original Boltz-2 runs. Check `boltz_rfdiffusion/complex/*.yaml` for the canonical sequence.

### Interpreting results
- **ipTM ≥ 0.70**: Strong interface confidence — submit to Adaptyv
- **ipTM 0.65–0.70**: Borderline — submit if slots allow
- **ipTM < 0.65**: Backbone geometry was good but ProteinMPNN couldn't find a good sequence → try partial diffusion or longer binder length

### Expected yield
From the 31 not-yet-submitted high-sc_proxy backbones × 9 sequences = 279 candidates.
At 20% pass rate → ~55 new sequences. Best-case 35–40 additional strong binders.

### Next steps if yield is low
1. Try higher MPNN temperature (0.20) on top 10 backbones
2. Run partial diffusion (noise 15–20 steps) on best 5 backbones to explore nearby backbone space
3. Consider longer binders (100–150 AA) for backbones where contacts are concentrated at a small patch